# 03 - Preprocessing

## Load the clean images

In [ ]:
import os
from PIL import Image
import numpy as np

BASE_PATH = os.path.join("..", "..", "datasets", "b_classification_flowers")
DATASET_PATH = os.path.join(BASE_PATH, "b_classification_flowers_clean")

IMG_SIZE = (128, 128)

X = []
y = []

classes = sorted(name for name in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, name)))

print("Classes:", classes)

for label, class_name in enumerate(classes):

    class_path = os.path.join(DATASET_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):

        img_path = os.path.join(class_path, img_name)

        try:
            with Image.open(img_path) as image:
                img = image.convert("RGB")
                img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)
                img = np.array(img, dtype=np.float32) / 255.0
        except (OSError, ValueError):
            continue

        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(classes))

## Flattening

In [ ]:
X_flat = X.reshape(X.shape[0], -1)

print("Original shape:", X.shape)
print("Flattened shape:", X_flat.shape)

### Train, Validation, and Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split training data and temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X_flat,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split validation and test data
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

## StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Testing:", X_test_scaled.shape)

## Save the processed data

In [ ]:
import pickle

BASE_PATH = os.path.join("..", "..", "datasets", "b_classification_flowers")
PROCESSED_PATH = os.path.join(BASE_PATH, "processed")
os.makedirs(PROCESSED_PATH, exist_ok=True)

np.savez(
    os.path.join(PROCESSED_PATH, "flowers_split.npz"),
    X_train=X_train_scaled,
    X_val=X_val_scaled,
    X_test=X_test_scaled,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test
)

with open(os.path.join(PROCESSED_PATH, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(PROCESSED_PATH, "classes.pkl"), "wb") as f:
    pickle.dump(classes, f)

print("Saved processed files to:", PROCESSED_PATH)